In [ ]:
# 1. KÜTÜPHANELERİ YÜKLE
import pandas as pd
from sklearn.metrics import mean_squared_error
import numpy as np
from catboost import CatBoostRegressor

In [2]:
# 2. VERİLERİ YÜKLE
# train.csv ve test.csv dosyalarını yükle
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
# 3. TARİH FORMATINI DÜZENLE
# 'event_time' sütununu datetime tipine çevirme
train_df['event_time'] = pd.to_datetime(train_df['event_time'])
test_df['event_time'] = pd.to_datetime(test_df['event_time'])

In [ ]:
# 4. TÜM VERİLERİ BİRLEŞTİR
# Özellik mühendisliği için train + test verilerini bir araya getirme
all_data = pd.concat([train_df, test_df], ignore_index=True)

In [5]:
# 5. OTURUM TEMELLİ ÖZELLİKLER
# Her bir oturumun başlangıç zamanı
all_data['session_start_time'] = all_data.groupby('user_session')['event_time'].transform('min')

# Her bir olayın oturum başlangıcına göre geçen süre (saniye cinsinden)
all_data['session_duration'] = (all_data['event_time'] - all_data['session_start_time']).dt.total_seconds()

In [ ]:
# 6. OLAY TİPLERİ SAYIMI
# Her bir user_session için event_type sayısı
event_counts = all_data.pivot_table(
    index='user_session',
    columns='event_type',
    values='event_time',
    aggfunc='count'
).fillna(0)

# Sütun isimlerini daha açıklayıcı yapma
event_counts.columns = [f'event_type_count_{col}' for col in event_counts.columns]
event_counts = event_counts.reset_index()

In [7]:
# 7. OTURUM ÖZELLİKLERİNİ TOPLAMA
session_features = all_data.groupby('user_session').agg(
    session_count=('event_type', 'count'),                # toplam event sayısı
    event_type_nunique=('event_type', 'nunique'),         # farklı event türü sayısı
    product_nunique=('product_id', 'nunique'),            # farklı ürün sayısı
    category_nunique=('category_id', 'nunique'),          # farklı kategori sayısı
    user_nunique=('user_id', 'nunique'),                  # farklı kullanıcı sayısı
    session_duration_max=('session_duration', 'max'),     # en uzun oturum süresi
    last_event_type=('event_type', lambda x: x.iloc[-1])  # son gerçekleşen event tipi
).reset_index()

In [ ]:
# 8. SAYIMLAR + OTURUM ÖZELLİKLERİ

# event_counts ve session_features birleştiriliyor
session_features = pd.merge(session_features, event_counts, on='user_session', how='left')

In [ ]:
# 9. TRAIN ve TEST'E ÖZELLİKLERİ EKLE
train_df_final = pd.merge(train_df, session_features, on='user_session', how='left')
test_df_final = pd.merge(test_df, session_features, on='user_session', how='left')

# Eksik değerleri doldurma
train_df_final.fillna(0, inplace=True)
test_df_final.fillna(0, inplace=True)

In [10]:
# 10. ÖZELLİK VE HEDEF TANIMLARI
# Kategorik sütunlar
categorical_features = ['user_id', 'product_id', 'category_id', 'last_event_type']

# Kullanılmayacak sütunlar
features_to_drop = ['event_time', 'user_session', 'session_start_time', 'session_value', 'event_type']

# Bağımsız değişkenler (X)
features = [col for col in train_df_final.columns if col not in features_to_drop]

# Bağımlı değişken (y)
target = 'session_value'

In [12]:
# 11. TRAIN / TEST AYRIMI
X_train = train_df_final[features]
y_train = train_df_final[target]
X_test = test_df_final[features]

# Hedef değişkenin log dönüşümü (büyük değerlerin etkisini azaltmak için)
y_train = np.log1p(y_train)

In [ ]:
# 12. CATBOOST MODEL PARAMETRELERİ
cat_params = {
    'iterations': 5000,             # ağaç sayısı (yüksek değer daha iyi ama yavaş)
    'learning_rate': 0.01,          # küçük öğrenme oranı
    'depth': 8,                     # ağaç derinliği
    'l2_leaf_reg': 3,               # regularization
    'loss_function': 'RMSE',        # regresyon kaybı
    'verbose': 1000,                # her 1000 adımda çıktı
    'random_seed': 42,              # tekrar üretilebilirlik
    'early_stopping_rounds': 300    # erken durdurma
}

# CatBoost için kategorik sütun indexlerini bul
cat_features_indices = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

In [14]:
# 13. MODELİ EĞİT
cat_model = CatBoostRegressor(**cat_params)
cat_model.fit(X_train, y_train, cat_features=cat_features_indices)

0:	learn: 0.9429059	total: 319ms	remaining: 26m 36s
1000:	learn: 0.3869375	total: 2m 3s	remaining: 8m 14s
2000:	learn: 0.3795860	total: 4m 6s	remaining: 6m 9s
3000:	learn: 0.3756973	total: 7m 19s	remaining: 4m 52s
4000:	learn: 0.3728255	total: 10m 42s	remaining: 2m 40s
4999:	learn: 0.3703117	total: 13m 50s	remaining: 0us


In [15]:
# 14. TAHMİNLER
# Test seti için tahminler
predictions = cat_model.predict(X_test)

# Log dönüşümü geri al
predictions = np.expm1(predictions)

# Negatif tahminleri 0'a sabitle
predictions[predictions < 0] = 0

In [ ]:
# 15. SUBMISSION DOSYASI
submission_df = pd.DataFrame({
    'user_session': test_df_final['user_session'],
    'session_value': predictions
})

# Aynı user_session için birden fazla tahmin varsa ilkini al
submission_df = submission_df.drop_duplicates(subset=['user_session'], keep='first')

# CSV olarak kaydet
submission_df.to_csv('submission.csv', index=False)
print("Yeni submission dosyası başarıyla oluşturuldu: submission.csv")

Yeni submission dosyası başarıyla oluşturuldu: submission.csv
